In [ ]:
import os
import sys

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/Stocks'
except ImportError:
    REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import logging
import warnings
import yfinance as yf
from IPython.display import HTML, display

# Silence yfinance's internal logger and noisy warnings so only genuine
# errors/warnings raised by this notebook surface.
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

# =========================================================================
# 0. SELECT WHICH ALLOCATION TO VIEW
# =========================================================================
# Default = AI (wave) allocation. Switch WITHOUT editing this notebook by
# setting a variable BEFORE running, e.g. in a driver cell:
#
#     portfolio_use = 'allocation'      # sector/satellite targets
#     run_notebook('portfolio_overview.ipynb')
#
#     portfolio_use = 'ai_allocation'   # AI value-chain wave targets (default)
#     run_notebook('portfolio_overview.ipynb')
#
# Accepted values (case-insensitive, aliases tolerated):
#   AI   : 'ai', 'ai_allocation', 'AI_allocations', 'wave'
#   SEC  : 'allocation', 'allocations', 'sector', 'satellite'
# Resolution order: explicit `portfolio_use` var  ->  env PORTFOLIO_USE  ->  'ai'.
try:
    _sel_raw = portfolio_use  # noqa: F821  (may be injected by a driver cell)
except NameError:
    _sel_raw = os.environ.get('PORTFOLIO_USE', 'ai')
_sel = str(_sel_raw).strip().lower()
_AI_ALIASES = {'ai', 'ai_allocation', 'ai_allocations', 'wave', 'waves'}
_SEC_ALIASES = {'allocation', 'allocations', 'sector', 'satellite', 'sectors'}
if _sel in _AI_ALIASES:
    ALLOCATION_MODE = 'ai'
elif _sel in _SEC_ALIASES:
    ALLOCATION_MODE = 'sector'
else:
    warnings.warn(f"Unrecognized portfolio_use={_sel_raw!r}; defaulting to AI allocation.")
    ALLOCATION_MODE = 'ai'

# Holdings (share counts) and ETF look-through describe the REAL portfolio and
# are independent of which target set we compare against — always load them
# from the sector module (the AI module is a target proposal with no holdings).
from portfolio.allocations import (
    my_current_shares, ETF_LOOK_THROUGH, MONTHLY_DEPOSIT,
)

# CHANGED: no stocks/ETFs are currently held — only crypto. Zero out the
# stock book so all stock & ETF values show $0 while targets/prices still
# render. Crypto holdings are kept as-is.
my_current_shares = {}
from portfolio.crypto import (
    CRYPTO_TARGET_WEIGHTS, my_current_crypto, MONTHLY_DEPOSIT_CHF,
)
from portfolio.helpers import get_price, get_live_fx_rate

# =========================================================================
# 0b. BUILD A UNIFIED VIEW OF THE SELECTED ALLOCATION
# =========================================================================
# Both allocations are normalized into the same shape so the rest of the
# notebook is structure-agnostic:
#   MACRO_TARGETS : {slice_key: target_weight}          (sums to 1.0)
#   BASKETS       : {slice_key: {ticker: sub_weight}}   (each sums to 1.0)
#   ETF_SLICES    : set of slice_keys that are directly-held ETFs (no basket)
#   MACRO_LABELS  : {slice_key: human label}
#   MACRO_COLORS  : {slice_key: bg color}
#   ALLOC_TITLE / ALLOC_SUBTITLE / BASKET_TAB_LABEL : presentation strings

if ALLOCATION_MODE == 'sector':
    from portfolio.allocations import (
        TARGET_WEIGHTS as _RAW_TARGETS,
        NUCLEAR_BASKET_TARGETS, QUANTUM_BASKET_TARGETS, CYBER_BASKET_TARGETS,
        INDUSTRIAL_BASKET_TARGETS, SPECGROWTH_BASKET_TARGETS, OTHER_BASKET_TARGETS,
    )
    _STRATEGY = {}  # sector taxonomy has no per-ticker strategy classification
    MACRO_TARGETS = dict(_RAW_TARGETS)
    ETF_SLICES = {'XAIX.DE', 'SMHV.SW', 'QDVE.DE'}
    BASKETS = {
        'NUCLEAR_SATELLITE':    NUCLEAR_BASKET_TARGETS,
        'QUANTUM_SATELLITE':    QUANTUM_BASKET_TARGETS,
        'CYBER_SATELLITE':      CYBER_BASKET_TARGETS,
        'INDUSTRIAL_SATELLITE': INDUSTRIAL_BASKET_TARGETS,
        'SPECGROWTH_SATELLITE': SPECGROWTH_BASKET_TARGETS,
        'OTHER_SATELLITE':      OTHER_BASKET_TARGETS,
    }
    MACRO_LABELS = {
        'XAIX.DE': 'XAIX.DE — AI & Big Data Index',
        'SMHV.SW': 'SMHV.SW — Semiconductors',
        'QDVE.DE': 'QDVE.DE — S&P 500 Info Tech',
        'NUCLEAR_SATELLITE': 'Nuclear Satellite (CCJ, GEV, SRUUF, LEU, SMR, OKLO, APLD)',
        'QUANTUM_SATELLITE': 'Quantum Satellite (IONQ, QNT, QBTS, RGTI, QUBT)',
        'CYBER_SATELLITE': 'Cyber Satellite (CRWD, PANW)',
        'INDUSTRIAL_SATELLITE': 'Industrial Satellite (BWXT, POWL, VRT, FIX)',
        'SPECGROWTH_SATELLITE': 'SpecGrowth Satellite (RKLB, LSCC, CRDO, VKTX)',
        'OTHER_SATELLITE': 'Other Satellite (TMDX, AXON, ENVX)',
    }
    MACRO_COLORS = {
        'XAIX.DE': '#E3F2FD', 'SMHV.SW': '#E3F2FD', 'QDVE.DE': '#E3F2FD',
        'NUCLEAR_SATELLITE': '#FFF8DC', 'QUANTUM_SATELLITE': '#F3E6F5',
        'CYBER_SATELLITE': '#FFEBEE', 'INDUSTRIAL_SATELLITE': '#E8EAF6',
        'SPECGROWTH_SATELLITE': '#E0F7FA', 'OTHER_SATELLITE': '#F1F8E9',
    }
    ALLOC_TITLE = 'Portfolio Overview — Sector / Satellite Allocation'
    ALLOC_SUBTITLE = 'Target set: portfolio/allocations.py (sector taxonomy)'
    BASKET_TAB_LABEL = 'Baskets'
    BASKET_TAB_HEADER = 'Satellite Baskets'

else:  # 'ai'  — AI value-chain wave allocation
    from portfolio.AI_allocations import (
        TARGET_WEIGHTS as _RAW_TARGETS,
        W1_SILICON_TARGETS, W2_POWER_TARGETS, W3_DCINFRA_TARGETS,
        W4_CLOUD_TARGETS, W5_SOFTWARE_TARGETS, W6_SPEC_TARGETS,
        STRATEGY as _STRATEGY,
    )
    MACRO_TARGETS = dict(_RAW_TARGETS)
    ETF_SLICES = set()  # AI allocation has no directly-held ETF slice
    BASKETS = {
        'W1_SILICON':  W1_SILICON_TARGETS,
        'W2_POWER':    W2_POWER_TARGETS,
        'W3_DCINFRA':  W3_DCINFRA_TARGETS,
        'W4_CLOUD':    W4_CLOUD_TARGETS,
        'W5_SOFTWARE': W5_SOFTWARE_TARGETS,
        'W6_SPEC':     W6_SPEC_TARGETS,
    }
    MACRO_LABELS = {
        'W1_SILICON':  'W1 Silicon (SMHV.SW, NVDA, AVGO, ASML, MRVL, TSM)',
        'W2_POWER':    'W2 Power (GEV, CEG, CCJ, POWL, OKLO, VST)',
        'W3_DCINFRA':  'W3 DC-Infra (VRT, ANET, CRDO, FIX, COHR)',
        'W4_CLOUD':    'W4 Cloud (MSFT, GOOGL, AMZN, META, ORCL)',
        'W5_SOFTWARE': 'W5 Software (PANW, CRWD, NOW, PLTR, SNOW, DDOG)',
        'W6_SPEC':     'W6 Speculative (AXON, TMDX, IONQ, RKLB)',
    }
    MACRO_COLORS = {
        'W1_SILICON': '#E3F2FD', 'W2_POWER': '#FFF8DC', 'W3_DCINFRA': '#E8EAF6',
        'W4_CLOUD': '#E6F4EA', 'W5_SOFTWARE': '#FFEBEE', 'W6_SPEC': '#F3E6F5',
    }
    ALLOC_TITLE = 'Portfolio Overview — AI Value-Chain (Wave) Allocation'
    ALLOC_SUBTITLE = 'Target set: portfolio/AI_allocations.py (W1 silicon -> W6 speculative)'
    BASKET_TAB_LABEL = 'Baskets'
    BASKET_TAB_HEADER = 'Wave Baskets'



# =========================================================================
# 0c. FORECAST MAP (ticker -> (min CAGR %, max CAGR %)) for the Growth tab
# =========================================================================
# AI mode uses the ticker-keyed WAVE_FORECASTS. Sector mode resolves the
# display-name-keyed stock_forecast_models / growth_forecast_models back to
# bare tickers via the assets.py name->ticker maps. Either way TICKER_CAGR
# ends up keyed by the same bare tickers the BASKETS use.
TICKER_CAGR = {}
try:
    if ALLOCATION_MODE == 'ai':
        from config.forecasts import WAVE_FORECASTS
        for _t, _m in WAVE_FORECASTS.items():
            TICKER_CAGR[_t] = (_m['min_rate'], _m['max_rate'])
    else:
        from config.assets import single_stocks as _ss, etfs as _etfs
        from config.forecasts import (
            stock_forecast_models as _sfm, growth_forecast_models as _gfm,
        )
        _name2t = {**_ss, **_etfs}
        for _name, _m in _sfm.items():
            _t = _name2t.get(_name)
            if _t:
                TICKER_CAGR[_t] = (_m['min_rate'], _m['max_rate'])
        for _name, _m in _gfm.items():
            _t = _name2t.get(_name)
            if _t:
                TICKER_CAGR[_t] = (_m['rate'], _m['rate'])
        # Explicit single-point CAGR for the sector core ETFs whose display
        # names don't round-trip to tickers via the assets maps.
        for _t, _r in {'XAIX.DE': 19.2, 'QDVE.DE': 15.6, 'SMHV.SW': 18.2}.items():
            TICKER_CAGR.setdefault(_t, (_r, _r))
except Exception as _exc:  # noqa: BLE001
    warnings.warn(f"Forecast map unavailable; Growth tab will be sparse: {_exc}")

# =========================================================================
# 1. FETCH LIVE PRICES
# =========================================================================
# Price every held ticker plus every ticker referenced by the selected
# allocation (AI targets may name stocks not yet held).
_alloc_tickers = set()
for _bk in BASKETS.values():
    _alloc_tickers.update(_bk.keys())
_alloc_tickers.update(ETF_SLICES)
_price_tickers = set(my_current_shares.keys()) | _alloc_tickers

import time

def _get_price_resilient(ticker, retries=3, delay=1.0):
    """get_price with retries so a transient curl/network hiccup (e.g. the
    intermittent 'Failure writing output' on QDVE.DE) self-heals instead of
    aborting the run. Returns 0.0 only after all attempts fail."""
    for attempt in range(retries):
        try:
            price = get_price(ticker)
            if price:  # non-zero, non-None
                return price
        except Exception:
            pass
        if attempt < retries - 1:
            time.sleep(delay)
    warnings.warn(f"Price unavailable for {ticker} after {retries} attempts; using 0.0")
    return 0.0

stock_prices = {t: _get_price_resilient(t) for t in _price_tickers}

def _get_crypto_price_resilient(ticker, retries=3, delay=1.0):
    for attempt in range(retries):
        try:
            data = yf.Ticker(ticker).history(period='1d')
            if not data.empty:
                return float(data['Close'].iloc[-1])
        except Exception:
            pass
        if attempt < retries - 1:
            time.sleep(delay)
    warnings.warn(f"Crypto price unavailable for {ticker} after {retries} attempts")
    return 0.01

crypto_prices_usd = {t: _get_crypto_price_resilient(t) for t in CRYPTO_TARGET_WEIGHTS}

USD_TO_CHF = get_live_fx_rate('USD', 'CHF')
EUR_TO_CHF = get_live_fx_rate('EUR', 'CHF')
CHF_TO_USD = 1 / USD_TO_CHF if USD_TO_CHF else 0
CHF_TO_EUR = 1 / EUR_TO_CHF if EUR_TO_CHF else 0

import base64, io
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def fx_sparkline_b64(ticker, color='#2C3E50'):
    try:
        data = yf.Ticker(ticker).history(period="1y")
        if data.empty: return ""
        closes = data['Close'].values
        fig, ax = plt.subplots(figsize=(2.2, 0.6))
        ax.plot(closes, color=color, linewidth=1.2)
        ax.fill_between(range(len(closes)), closes, alpha=0.1, color=color)
        y_min, y_max = closes.min(), closes.max()
        y_pad = (y_max - y_min) * 0.05 if y_max != y_min else 0.01
        ax.set_ylim(y_min - y_pad, y_max + y_pad)
        ax.axis('off')
        fig.patch.set_alpha(0)
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=80, bbox_inches='tight', pad_inches=0.02, transparent=True)
        plt.close(fig)
        buf.seek(0)
        return base64.b64encode(buf.read()).decode('utf-8')
    except Exception:
        return ""

chf_usd_spark = fx_sparkline_b64('CHFUSD=X', '#1565C0')
chf_eur_spark = fx_sparkline_b64('CHFEUR=X', '#1B5E20')

# =========================================================================
# 2. COMPUTE PORTFOLIO VALUES (generic)
# =========================================================================
stock_values = {t: shares * stock_prices.get(t, 0) for t, shares in my_current_shares.items()}

# Per-slice held value: ETF slices use the slice ticker itself; basket slices
# sum the value of their member tickers actually held.
macro_values = {}
for slice_key in MACRO_TARGETS:
    if slice_key in ETF_SLICES:
        macro_values[slice_key] = stock_values.get(slice_key, 0)
    elif slice_key in BASKETS:
        macro_values[slice_key] = sum(stock_values.get(t, 0) for t in BASKETS[slice_key])
    else:
        macro_values[slice_key] = stock_values.get(slice_key, 0)

# Any held value not captured by the selected allocation (e.g. sector holdings
# that aren't part of the AI waves) is shown as an explicit residual so the
# macro table still totals the full stock book.
_mapped_tickers = set()
for slice_key in MACRO_TARGETS:
    if slice_key in ETF_SLICES:
        _mapped_tickers.add(slice_key)
    elif slice_key in BASKETS:
        _mapped_tickers.update(BASKETS[slice_key].keys())
    else:
        _mapped_tickers.add(slice_key)
_unmapped_val = sum(v for t, v in stock_values.items() if t not in _mapped_tickers)
UNMAPPED_KEY = '_UNMAPPED'
if _unmapped_val > 1e-6:
    macro_values[UNMAPPED_KEY] = _unmapped_val
    MACRO_LABELS[UNMAPPED_KEY] = 'Other holdings (not in selected allocation)'
    MACRO_COLORS[UNMAPPED_KEY] = '#ECEFF1'

total_stock_val = sum(macro_values.values())

crypto_values_usd = {
    c: my_current_crypto.get(c, 0) * crypto_prices_usd.get(c, 0)
    for c in CRYPTO_TARGET_WEIGHTS
}
total_crypto_usd = sum(crypto_values_usd.values())
total_crypto_chf = total_crypto_usd * USD_TO_CHF
total_portfolio_usd = total_stock_val + total_crypto_usd
total_stock_chf = total_stock_val * USD_TO_CHF
total_portfolio_chf = total_portfolio_usd * USD_TO_CHF

# =========================================================================
# 3. ETF LOOK-THROUGH EXPOSURE (generic, recursive)
# =========================================================================
# Net underlying exposure resolves every slice down to real underlying
# stocks. Any ticker that is itself an ETF with a look-through definition
# (e.g. SMHV.SW inside the AI W1 basket) is EXPANDED into its constituents
# instead of being shown as a single ETF line — so SMHV.SW disappears and
# its weight is distributed across MU/AMD/AVGO/... per ETF_LOOK_THROUGH.
exposure = {}
exposure_sources = {}

def _add_exposure(ticker, weight, source):
    """Add net weight for a ticker. If the ticker has an ETF_LOOK_THROUGH
    definition, recurse into its holdings; otherwise record it directly."""
    if weight <= 0:
        return
    if ticker in ETF_LOOK_THROUGH:
        for sub, sub_w in ETF_LOOK_THROUGH[ticker].items():
            _add_exposure(sub, weight * sub_w, source)
    else:
        exposure[ticker] = exposure.get(ticker, 0) + weight
        exposure_sources.setdefault(ticker, []).append(source)

# Directly-held ETF macro slices (sector mode: XAIX/SMHV/QDVE held outright).
for etf in ETF_LOOK_THROUGH:
    macro_weight = MACRO_TARGETS.get(etf, 0)
    if macro_weight > 0 and etf in ETF_SLICES:
        _add_exposure(etf, macro_weight, etf)

# Basket members (expands any ETF member such as SMHV.SW in AI's W1).
for slice_key, basket in BASKETS.items():
    slice_weight = MACRO_TARGETS.get(slice_key, 0)
    src_label = slice_key.replace('_SATELLITE', '')
    for stock, sub_weight in basket.items():
        _add_exposure(stock, slice_weight * sub_weight, src_label)

sorted_exposure = sorted(exposure.items(), key=lambda x: x[1], reverse=True)

# =========================================================================
# 4. HELPERS
# =========================================================================
def fmt_money(val):
    if abs(val) >= 1e6: return f'${val/1e6:,.2f}M'
    elif abs(val) >= 1e3: return f'${val:,.0f}'
    return f'${val:,.2f}'

def fmt_chf(val):
    if abs(val) >= 1e6: return f'CHF {val/1e6:,.2f}M'
    elif abs(val) >= 1e3: return f'CHF {val:,.0f}'
    return f'CHF {val:,.2f}'

def drift_cell(current_pct, target_pct):
    diff = current_pct - target_pct
    if abs(diff) < 2: color = '#1B5E20'
    elif abs(diff) < 5: color = '#E65100'
    else: color = '#B71C1C'
    return f'<td style="color:{color}; font-weight:bold;">{diff:+.1f}%</td>'

CRYPTO_COLORS = {
    'BTC-USD': '#FFF3E0', 'ETH-USD': '#E3F2FD', 'SOL-USD': '#F3E6F5',
    'RENDER-USD': '#E8F5E9', 'LINK-USD': '#E3F2FD', 'XRP-USD': '#ECEFF1',
}
CRYPTO_LABELS = {
    'BTC-USD': 'Bitcoin', 'ETH-USD': 'Ethereum', 'SOL-USD': 'Solana',
    'RENDER-USD': 'Render', 'LINK-USD': 'Chainlink', 'XRP-USD': 'XRP',
}

# Source colors/labels: known ETF + generic slice fallback.
SOURCE_COLORS = {
    'SMHV.SW': '#E6F0FA', 'QDVE.DE': '#E6F7F9', 'XAIX.DE': '#E6F4EA',
    'NUCLEAR': '#FCF7E6', 'QUANTUM': '#FAE6FA', 'CYBER': '#FCE8E6',
    'INDUSTRIAL': '#E8EAF6', 'SPECGROWTH': '#E0F7FA', 'OTHER': '#F1F8E9',
}
def source_label(src):
    etf_names = {
        'SMHV.SW': 'Core Semiconductors (SMHV.SW)',
        'QDVE.DE': 'S&P 500 Info Tech (QDVE.DE)',
        'XAIX.DE': 'AI & Big Data Index (XAIX.DE)',
    }
    if src in etf_names:
        return etf_names[src]
    return MACRO_LABELS.get(src, f'Slice ({src})')

# =========================================================================
# 5. BUILD HTML
# =========================================================================
html = []
html.append("""<style>
.po-tab { overflow: hidden; border: 1px solid #000; background-color: #f1f1f1; margin-top: 10px; }
.po-tab button { background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 14px 16px; transition: 0.3s; color: black; font-weight: bold; font-size: 14px; }
.po-tab button:hover { background-color: #ddd; }
.po-tab button.active { background-color: #ccc; }
.po-tabcontent { display: none; padding: 20px; border: 1px solid #000; border-top: none; background: white; }
.po-subtab { overflow: hidden; background-color: #e9e9e9; border: 1px solid #000; border-bottom: none; margin-top: 6px; }
.po-subtab button { background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 10px 14px; transition: 0.3s; color: black; font-weight: bold; font-size: 13px; }
.po-subtab button:hover { background-color: #d5d5d5; }
.po-subtab button.active { background-color: #bbb; }
.po-subcontent { display: none; padding: 14px; border: 1px solid #000; border-top: none; background: white; }
.po-table { border-collapse: collapse; width: 100%; font-family: Arial, sans-serif; font-size: 12px; }
.po-table th { background: #2C3E50; color: white; padding: 8px 12px; text-align: left; font-weight: bold; }
.po-table td { padding: 6px 12px; border-bottom: 1px solid #ddd; color: #1a1a1a; }
.po-table tr:hover { filter: brightness(0.95); }
.po-total td { background: #2C3E50 !important; color: white !important; font-weight: bold; }
.po-title { font-size: 18px; font-weight: bold; color: white; background: #2C3E50; padding: 12px 16px; border-radius: 6px 6px 0 0; }
.po-sub { font-size: 12px; color: #ccc; background: #2C3E50; padding: 0 16px 10px; border-radius: 0 0 6px 6px; margin-bottom: 14px; }
.po-header { font-size: 16px; font-weight: bold; color: white; background: #2C3E50; padding: 10px 16px; border-radius: 6px 6px 0 0; margin-top: 20px; }
.po-summary { display: flex; gap: 20px; margin-bottom: 20px; flex-wrap: wrap; }
.po-card { background: white; padding: 16px 24px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); text-align: center; min-width: 180px; }
.po-card-value { font-size: 24px; font-weight: bold; color: #2C3E50; }
.po-card-label { font-size: 12px; color: #555; margin-top: 4px; }
</style>
<script>
function poOpenTab(evt, tabName) {
  var i, tc, tl;
  tc = document.getElementsByClassName('po-tabcontent');
  for (i = 0; i < tc.length; i++) { tc[i].style.display = 'none'; }
  tl = document.getElementsByClassName('po-tablink');
  for (i = 0; i < tl.length; i++) { tl[i].className = tl[i].className.replace(' active', ''); }
  document.getElementById(tabName).style.display = 'block';
  evt.currentTarget.className += ' active';
}
function poOpenSub(evt, subName, contentClass, linkClass) {
  var i, sc, sl;
  sc = document.getElementsByClassName(contentClass);
  for (i = 0; i < sc.length; i++) { sc[i].style.display = 'none'; }
  sl = document.getElementsByClassName(linkClass);
  for (i = 0; i < sl.length; i++) { sl[i].className = sl[i].className.replace(' active', ''); }
  document.getElementById(subName).style.display = 'block';
  evt.currentTarget.className += ' active';
}
</script>""")

import datetime as _dt
html.append('<div class="po-title">Portfolio Overview</div>')

stock_pct = total_stock_val / total_portfolio_usd * 100 if total_portfolio_usd > 0 else 0
crypto_pct = total_crypto_usd / total_portfolio_usd * 100 if total_portfolio_usd > 0 else 0
html.append('<div class="po-summary">')
html.append(f'<div class="po-card"><div class="po-card-value">{fmt_chf(total_portfolio_chf)}</div><div class="po-card-label">Total Portfolio (approx CHF)</div></div>')
html.append(f'<div class="po-card"><div class="po-card-value">{fmt_money(total_stock_val)}</div><div class="po-card-label">Stocks & ETFs ({stock_pct:.0f}%)</div></div>')
html.append(f'<div class="po-card"><div class="po-card-value">{fmt_money(total_crypto_usd)}</div><div class="po-card-label">Crypto ({crypto_pct:.0f}%)</div></div>')
chf_usd_img = f'<img src="data:image/png;base64,{chf_usd_spark}" style="display:block;">' if chf_usd_spark else ''
chf_eur_img = f'<img src="data:image/png;base64,{chf_eur_spark}" style="display:block;">' if chf_eur_spark else ''
html.append(f'<div class="po-card" style="background:#2C3E50;"><div style="display:flex; align-items:center; gap:12px;"><div style="text-align:left;"><div class="po-card-value" style="color:white;">{CHF_TO_USD:.4f}</div><div class="po-card-label" style="color:white;">CHF/USD</div></div>{chf_usd_img}</div></div>')
html.append(f'<div class="po-card" style="background:#2C3E50;"><div style="display:flex; align-items:center; gap:12px;"><div style="text-align:left;"><div class="po-card-value" style="color:white;">{CHF_TO_EUR:.4f}</div><div class="po-card-label" style="color:white;">CHF/EUR</div></div>{chf_eur_img}</div></div>')
html.append('</div>')

html.append('<div class="po-tab">')
html.append('<button class="po-tablink active" onclick="poOpenTab(event, \'po_macro\')">Macro Allocation</button>')
html.append(f'<button class="po-tablink" onclick="poOpenTab(event, \'po_baskets\')">{BASKET_TAB_LABEL}</button>')
html.append('<button class="po-tablink" onclick="poOpenTab(event, \'po_growth\')">Growth</button>')
if _STRATEGY:
    html.append('<button class="po-tablink" onclick="poOpenTab(event, \'po_category\')">Category</button>')
if _STRATEGY:
    html.append('<button class="po-tablink" onclick="poOpenTab(event, \'po_bottleneck\')">Bottlenecks</button>')
html.append('<button class="po-tablink" onclick="poOpenTab(event, \'po_exposure\')">Exposure</button>')
html.append('<button class="po-tablink" onclick="poOpenTab(event, \'po_crypto\')">Crypto</button>')
html.append('<button class="po-tablink" onclick="poOpenTab(event, \'po_rationale\')">Rationale</button>')
html.append('</div>')

# --- TAB 1: MACRO ALLOCATION ---
html.append('<div id="po_macro" class="po-tabcontent" style="display:block;">')
html.append('<div class="po-header">Macro Allocation</div>')
html.append('<table class="po-table">')
html.append('<tr><th>Asset / Slice</th><th>Value</th><th>Current %</th><th>Target %</th><th>Drift</th></tr>')
# Iterate target slices, then append the residual row if present.
_macro_rows = list(MACRO_TARGETS.items())
if UNMAPPED_KEY in macro_values:
    _macro_rows.append((UNMAPPED_KEY, 0.0))
for asset, target_pct in _macro_rows:
    val = macro_values.get(asset, 0)
    cur_pct = val / total_stock_val * 100 if total_stock_val > 0 else 0
    tgt_pct = target_pct * 100
    label = MACRO_LABELS.get(asset, asset)
    bg = MACRO_COLORS.get(asset, '#FFFFFF')
    if asset in ETF_SLICES:
        label = f'<a href="https://finance.yahoo.com/quote/{asset}/" target="_blank" style="color:#1565C0;">{label}</a>'
    html.append(f'<tr style="background:{bg};"><td><b>{label}</b></td><td>{fmt_money(val)}</td><td>{cur_pct:.1f}%</td><td>{tgt_pct:.0f}%</td>{drift_cell(cur_pct, tgt_pct)}</tr>')
html.append(f'<tr class="po-total"><td>TOTAL</td><td>{fmt_money(total_stock_val)}</td><td>100%</td><td>100%</td><td>—</td></tr>')
html.append('</table>')
if total_stock_val > 0:
    chosen_stock = max(MACRO_TARGETS, key=lambda a: MACRO_TARGETS[a] - (macro_values.get(a, 0) / total_stock_val))
    html.append(f'<p style="margin-top:12px; font-size:14px; color:#1a1a1a;"><b>Most underweight slice:</b> {MACRO_LABELS.get(chosen_stock, chosen_stock)} — direct \u20ac{MONTHLY_DEPOSIT:,.0f} here</p>')
html.append('</div>')

# --- TAB 2: BASKETS (sector satellites OR AI waves) ---
html.append('<div id="po_baskets" class="po-tabcontent">')
html.append(f'<div class="po-header">{BASKET_TAB_HEADER}</div>')
html.append('<table class="po-table">')
html.append('<tr><th>Ticker</th><th>Basket</th><th>Price</th><th>Value</th><th>Basket %</th><th>Target %</th><th>Drift</th></tr>')
# Directly-held ETF slices (sector allocation: XAIX/SMHV/QDVE). They are macro
# slices with no sub-basket, so list each as its own single-line "basket" here
# too. AI allocation has no ETF slices, so this loop is a no-op there.
for etf in MACRO_TARGETS:
    if etf not in ETF_SLICES:
        continue
    price = stock_prices.get(etf, 0)
    val = macro_values.get(etf, 0)
    tgt_pct = MACRO_TARGETS.get(etf, 0) * 100
    bg = MACRO_COLORS.get(etf, '#FFFFFF')
    yf_link = f'<a href="https://finance.yahoo.com/quote/{etf}/" target="_blank" style="color:#1565C0;">{etf}</a>'
    html.append(f'<tr style="background:{bg};"><td><b>{yf_link}</b></td><td>Core ETF</td><td>${price:,.2f}</td><td>{fmt_money(val)}</td><td>100.0%</td><td>{tgt_pct:.0f}%</td>{drift_cell(100.0, 100.0)}</tr>')
for slice_key, targets in BASKETS.items():
    basket_total = macro_values.get(slice_key, 0)
    basket_name = MACRO_LABELS.get(slice_key, slice_key).split(' (')[0]
    bg = MACRO_COLORS.get(slice_key, '#FFFFFF')
    for ticker, target in targets.items():
        price = stock_prices.get(ticker, 0)
        val = stock_values.get(ticker, 0)
        cur_pct = val / basket_total * 100 if basket_total > 0 else 0
        tgt_pct = target * 100
        yf_link = f'<a href="https://finance.yahoo.com/quote/{ticker}/" target="_blank" style="color:#1565C0;">{ticker}</a>'
        html.append(f'<tr style="background:{bg};"><td><b>{yf_link}</b></td><td>{basket_name}</td><td>${price:,.2f}</td><td>{fmt_money(val)}</td><td>{cur_pct:.1f}%</td><td>{tgt_pct:.0f}%</td>{drift_cell(cur_pct, tgt_pct)}</tr>')
html.append('</table></div>')

# --- TAB 3: GROWTH (potential total return per basket/wave) ---
# For each basket, take the sub-weighted average of its tickers' min/max CAGR
# (only over tickers that have a forecast, re-normalizing their weights), then
# compound to 1y and 5y total returns. Also show a portfolio-level roll-up
# weighted by the macro (wave/slice) target weights.
def _fmt_pct(v):
    return f'+{v:.0f}%' if v >= 0 else f'{v:.0f}%'

def _basket_cagr(slice_key):
    """Sub-weighted (min,max) CAGR % for one basket, or None if no forecasts."""
    targets = BASKETS.get(slice_key)
    if not targets:
        # ETF macro slice (sector mode): single ticker == slice_key
        if slice_key in TICKER_CAGR:
            lo, hi = TICKER_CAGR[slice_key]
            return lo, hi
        return None
    num_lo = num_hi = wsum = 0.0
    for tk, wt in targets.items():
        if tk in TICKER_CAGR and wt > 0:
            lo, hi = TICKER_CAGR[tk]
            num_lo += lo * wt
            num_hi += hi * wt
            wsum += wt
    if wsum == 0:
        return None
    return num_lo / wsum, num_hi / wsum

html.append('<div id="po_growth" class="po-tabcontent">')
html.append('<div class="po-header">Potential Growth</div>')
html.append('<p style="font-size:13px; color:#555; max-width:820px;">'
            'Forward total-return estimates from CAGR forecasts (config/forecasts.py), '
            'compounded to 1y and 5y. Ranges are illustrative model outputs, not '
            'guarantees — wider bands mean more uncertain / speculative.</p>')

# Sub-tabs: Baskets (per wave/sleeve) and Stocks (per individual holding).
_grp_word = 'Wave' if ALLOCATION_MODE == 'ai' else 'Sleeve'
html.append('<div class="po-subtab">')
html.append('<button class="po-grow-sublink active" '
            "onclick=\"poOpenSub(event, 'po_grow_baskets', 'po-grow-subcontent', 'po-grow-sublink')\">"
            f'{BASKET_TAB_LABEL}</button>')
html.append('<button class="po-grow-sublink" '
            "onclick=\"poOpenSub(event, 'po_grow_stocks', 'po-grow-subcontent', 'po-grow-sublink')\">"
            'Stocks</button>')
html.append('</div>')

# --- SUB-TAB A: BASKETS (per wave/sleeve roll-up) ---
html.append('<div id="po_grow_baskets" class="po-grow-subcontent" style="display:block;">')
html.append('<table class="po-table">')
html.append(f'<tr><th>{_grp_word}</th><th>Target %</th><th>CAGR (lo-hi)</th>'
            '<th>1y Return</th><th>5y Return</th></tr>')

_port_lo_1y = _port_hi_1y = _port_lo_5y = _port_hi_5y = 0.0
_port_wsum = 0.0
for slice_key in list(MACRO_TARGETS.keys()):
    cagr = _basket_cagr(slice_key)
    tgt = MACRO_TARGETS.get(slice_key, 0)
    tgt_pct = tgt * 100
    label = MACRO_LABELS.get(slice_key, slice_key).split(' (')[0]
    bg = MACRO_COLORS.get(slice_key, '#FFFFFF')
    if cagr is None:
        html.append(f'<tr style="background:{bg};"><td><b>{label}</b></td>'
                    f'<td>{tgt_pct:.0f}%</td><td>—</td><td>—</td><td>—</td></tr>')
        continue
    lo, hi = cagr
    r1_lo = ((1 + lo / 100) ** 1 - 1) * 100
    r1_hi = ((1 + hi / 100) ** 1 - 1) * 100
    r5_lo = ((1 + lo / 100) ** 5 - 1) * 100
    r5_hi = ((1 + hi / 100) ** 5 - 1) * 100
    _port_lo_1y += r1_lo * tgt
    _port_hi_1y += r1_hi * tgt
    _port_lo_5y += r5_lo * tgt
    _port_hi_5y += r5_hi * tgt
    _port_wsum += tgt
    html.append(
        f'<tr style="background:{bg};"><td><b>{label}</b></td>'
        f'<td>{tgt_pct:.0f}%</td>'
        f'<td>{lo:.0f}%-{hi:.0f}%</td>'
        f'<td>{_fmt_pct(r1_lo)} to {_fmt_pct(r1_hi)}</td>'
        f'<td>{_fmt_pct(r5_lo)} to {_fmt_pct(r5_hi)}</td></tr>'
    )
if _port_wsum > 0:
    pl1, ph1 = _port_lo_1y / _port_wsum, _port_hi_1y / _port_wsum
    pl5, ph5 = _port_lo_5y / _port_wsum, _port_hi_5y / _port_wsum
    html.append(
        f'<tr class="po-total"><td>PORTFOLIO</td><td>100%</td><td>—</td>'
        f'<td>{_fmt_pct(pl1)} to {_fmt_pct(ph1)}</td>'
        f'<td>{_fmt_pct(pl5)} to {_fmt_pct(ph5)}</td></tr>'
    )
html.append('</table>')
html.append('<p style="font-size:11px; color:#888; margin-top:10px;">'
            f'{_grp_word} CAGR is the sub-weighted blend of its holdings; '
            '1y / 5y compound that band. PORTFOLIO row is the target-weighted '
            f'blend of the per-{_grp_word.lower()} returns.</p>')
html.append('</div>')

# --- SUB-TAB B: STOCKS (per individual holding) ---
html.append('<div id="po_grow_stocks" class="po-grow-subcontent">')
html.append('<table class="po-table">')
html.append(f'<tr><th>Ticker</th><th>{_grp_word}</th><th>CAGR (lo-hi)</th>'
            '<th>1y Return</th><th>5y Return</th></tr>')
# Order stocks by their wave/sleeve, then by descending sub-weight.
for slice_key in list(MACRO_TARGETS.keys()):
    targets = BASKETS.get(slice_key)
    label = MACRO_LABELS.get(slice_key, slice_key).split(' (')[0]
    bg = MACRO_COLORS.get(slice_key, '#FFFFFF')
    if not targets:
        # ETF macro slice (sector mode): single ticker == slice_key
        if slice_key in TICKER_CAGR:
            members = [(slice_key, 1.0)]
        else:
            continue
    else:
        members = sorted(targets.items(), key=lambda kv: -kv[1])
    for tk, _sw in members:
        if tk not in TICKER_CAGR:
            html.append(f'<tr style="background:{bg};"><td><b>{tk}</b></td>'
                        f'<td>{label}</td><td>—</td><td>—</td><td>—</td></tr>')
            continue
        lo, hi = TICKER_CAGR[tk]
        r1_lo = ((1 + lo / 100) ** 1 - 1) * 100
        r1_hi = ((1 + hi / 100) ** 1 - 1) * 100
        r5_lo = ((1 + lo / 100) ** 5 - 1) * 100
        r5_hi = ((1 + hi / 100) ** 5 - 1) * 100
        html.append(
            f'<tr style="background:{bg};"><td><b>{tk}</b></td>'
            f'<td>{label}</td>'
            f'<td>{lo:.0f}%-{hi:.0f}%</td>'
            f'<td>{_fmt_pct(r1_lo)} to {_fmt_pct(r1_hi)}</td>'
            f'<td>{_fmt_pct(r5_lo)} to {_fmt_pct(r5_hi)}</td></tr>'
        )
html.append('</table>')
html.append('<p style="font-size:11px; color:#888; margin-top:10px;">'
            'Per-holding CAGR bands (config/forecasts.py), compounded to 1y / 5y. '
            'Wider bands = more speculative.</p>')
html.append('</div>')

html.append('</div>')

# --- TAB: CATEGORY (holdings grouped by strategy: DCA -> Cycle -> Catalyst) ---
# Only meaningful when the allocation defines per-ticker strategies (AI mode).
if _STRATEGY:
    # net portfolio weight per ticker = direct basket weight * macro (wave) weight
    _net_w = {}
    for _sk, _bk in BASKETS.items():
        _mw = MACRO_TARGETS.get(_sk, 0)
        for _tk, _sw in _bk.items():
            _net_w[_tk] = _net_w.get(_tk, 0) + _sw * _mw

    _STRAT_ORDER = [
        ('dca',      'DCA',      'Buy monthly, hold forever, average down OK'),
        ('cycle',    'Cycle',    'Buy dips, trim into the cycle peak (~2027-29)'),
        ('catalyst', 'Catalyst', 'Buy once on the thesis, never average down, sell at the event'),
    ]
    _strat_color = {'dca': '#E6F4EA', 'cycle': '#FFF8DC', 'catalyst': '#FFEBEE'}

    html.append('<div id="po_category" class="po-tabcontent">')
    html.append('<div class="po-header">Holdings by Strategy</div>')
    html.append('<p style="font-size:13px; color:#555; max-width:820px;">'
                'Every holding grouped by its playbook: DCA (hold-forever compounders), '
                'Cycle (buy dips / sell peaks), then Catalyst (binary event bets). '
                'Net weight is the share of the whole portfolio.</p>')

    for _strat, _label, _desc in _STRAT_ORDER:
        _members = sorted(
            ((t, s) for t, s in _STRATEGY.items() if s == _strat),
            key=lambda kv: -_net_w.get(kv[0], 0),
        )
        if not _members:
            continue
        _bg = _strat_color.get(_strat, '#FFFFFF')
        html.append(f'<div class="po-header" style="font-size:14px;">'
                    f'{_label} ({len(_members)})</div>')
        html.append(f'<p style="font-size:12px; color:#666; margin:4px 0 8px;">{_desc}</p>')
        html.append('<table class="po-table">')
        html.append('<tr><th>Ticker</th><th>Net %</th>'
                    '<th>CAGR (lo-hi)</th><th>1y Return</th><th>5y Return</th></tr>')
        _grp_net = 0.0
        for _tk, _s in _members:
            _nw = _net_w.get(_tk, 0) * 100
            _grp_net += _nw
            if _tk in TICKER_CAGR:
                lo, hi = TICKER_CAGR[_tk]
                r1_lo = ((1 + lo / 100) ** 1 - 1) * 100
                r1_hi = ((1 + hi / 100) ** 1 - 1) * 100
                r5_lo = ((1 + lo / 100) ** 5 - 1) * 100
                r5_hi = ((1 + hi / 100) ** 5 - 1) * 100
                _cagr = f'{lo:.0f}%-{hi:.0f}%'
                _r1 = f'{_fmt_pct(r1_lo)} to {_fmt_pct(r1_hi)}'
                _r5 = f'{_fmt_pct(r5_lo)} to {_fmt_pct(r5_hi)}'
            else:
                _cagr = _r1 = _r5 = '—'
            html.append(
                f'<tr style="background:{_bg};"><td><b>{_tk}</b></td>'
                f'<td>{_nw:.2f}%</td><td>{_cagr}</td>'
                f'<td>{_r1}</td><td>{_r5}</td></tr>'
            )
        html.append(
            f'<tr class="po-total"><td>{_label} TOTAL</td>'
            f'<td>{_grp_net:.2f}%</td><td>—</td><td>—</td><td>—</td></tr>'
        )
        html.append('</table>')
    html.append('</div>')

if _STRATEGY:
    # --- TAB: BOTTLENECKS (AI supply-chain bottleneck-migration map) ---
    # Curated map of the AI bottleneck migration (compute -> HBM/packaging ->
    # power generation -> transmission -> cooling/water -> storage/permits).
    # Each row: ticker, bottleneck stage, cycle position, 1y & 2y price return,
    # and whether it sits in the user's allocation (and which wave/sleeve).
    # Rows include both HELD names and not-yet-held candidates so the migration
    # is visible end to end.
    #
    # stage  : the bottleneck layer (what is hard to get)
    # pos    : cycle position — Early / Mid / Late / Binary
    # (ticker, stage, pos)
    BOTTLENECK_MAP = [
        # --- Silicon: compute + HBM + advanced packaging (the original bottleneck) ---
        ("NVDA", "Compute (GPU)",              "Mid"),
        ("AMD",  "Compute (GPU #2)",           "Mid"),
        ("AVGO", "Custom ASIC",                "Mid"),
        ("MRVL", "Custom ASIC / optical",      "Mid"),
        ("MU",   "Memory / HBM",               "Mid"),
        ("TSM",  "Foundry + CoWoS packaging",  "Mid"),
        ("ASML", "EUV lithography",            "Mid"),
        ("BESI", "Advanced packaging (CoWoS)", "Early"),
        # --- Power: generation ("make electrons") — current frontier, maturing ---
        ("GEV",  "Power generation (turbines+grid)", "Late"),
        ("CEG",  "Power generation (nuclear)",       "Late"),
        ("VST",  "Power generation (merchant)",      "Late"),
        ("CCJ",  "Nuclear fuel (uranium)",           "Mid"),
        ("OKLO", "Power generation (SMR)",           "Binary"),
        # --- Power: transmission ("move electrons") — the NEXT frontier ---
        ("ETN",  "Transmission / electrification",   "Early"),
        ("PWR",  "Builds transmission lines",        "Early"),
        ("HUBB", "Transformers / grid gear",         "Early"),
        ("ABBN.SW", "HVDC / transformers (EU)",      "Early"),
        ("SIE.DE",  "Grid / transmission (EU)",      "Mid"),
        ("POWL", "Switchgear (building)",            "Late"),
        # --- DC-infra: cooling + heat rejection + water (next sub-bottleneck) ---
        ("VRT",  "Liquid cooling + power",           "Mid"),
        ("PNR",  "Water / thermal",                  "Early"),
        ("ANET", "DC networking",                    "Mid"),
        ("CRDO", "Optical interconnect",             "Mid"),
        ("COHR", "Optical components",               "Mid"),
        ("FIX",  "DC construction (labor/permits)",  "Late"),
    ]

    # Where does each ticker live in the user's allocation? Map ticker -> wave/sleeve.
    _ticker_wave = {}
    for _sk, _bk in BASKETS.items():
        _wlabel = MACRO_LABELS.get(_sk, _sk).split(' (')[0]
        for _tk in _bk:
            _ticker_wave[_tk] = _wlabel

    import pandas as pd

    # Resilient history fetch for 1y / 2y total price return.
    def _hist_returns(ticker):
        """Return (ret_1y_pct, ret_2y_pct); None entries if data unavailable."""
        for _attempt in range(2):
            try:
                h = yf.Ticker(ticker).history(period='3y', auto_adjust=True)['Close'].dropna()
                if len(h) < 2:
                    return (None, None)
                last = float(h.iloc[-1])
                def _r(days):
                    cutoff = h.index.max() - pd.Timedelta(days=days)
                    past = h[h.index <= cutoff]
                    if past.empty:
                        return None
                    return (last / float(past.iloc[-1]) - 1) * 100
                return (_r(365), _r(730))
            except Exception:
                time.sleep(0.5)
        return (None, None)

    _pos_color = {'Early': '#E6F4EA', 'Mid': '#FFF8DC', 'Late': '#FFE0CC', 'Binary': '#FFEBEE'}
    _pos_note = {
        'Early':  'Shortage just ramping — best entry, accumulate',
        'Mid':    'Shortage in full swing — hold, still climbing',
        'Late':   'Priced-for-perfection — trim on strength, watch signals',
        'Binary': 'Pre-revenue / event-driven — tiny size only',
    }

    def _fmt_ret(v):
        if v is None:
            return '<span style="color:#bbb;">n/a</span>'
        return _fmt_pct(v)

    html.append('<div id="po_bottleneck" class="po-tabcontent">')
    html.append('<div class="po-header">AI Bottleneck-Migration Map</div>')
    html.append('<p style="font-size:13px; color:#555; max-width:880px;">'
                'The AI buildout constraint keeps migrating down the stack: '
                '<b>compute -> HBM / packaging -> power generation -> transmission '
                '-> cooling / water -> storage / permits</b>. Each stage has its own '
                'cycle position; this maps every key name (held and candidate) onto '
                'that migration with 1y / 2y price returns and where it sits in your '
                'allocation.</p>')

    # Cycle-position legend
    html.append('<div style="margin:8px 0 14px; font-size:12px;">')
    for _p in ['Early', 'Mid', 'Late', 'Binary']:
        html.append(f'<span style="background:{_pos_color[_p]}; padding:3px 8px; '
                    f'border:1px solid #ccc; margin-right:6px; border-radius:3px;">'
                    f'<b>{_p}</b> — {_pos_note[_p]}</span>')
    html.append('</div>')

    html.append('<table class="po-table">')
    html.append('<tr><th>Ticker</th><th>Bottleneck stage</th><th>Cycle position</th>'
                '<th>1y Return</th><th>2y Return</th><th>In portfolio?</th></tr>')
    for _tk, _stage, _pos in BOTTLENECK_MAP:
        _r1, _r2 = _hist_returns(_tk)
        _bg = _pos_color.get(_pos, '#FFFFFF')
        _wave = _ticker_wave.get(_tk)
        if _wave:
            _inport = f'<b style="color:#1a7f37;">{_wave}</b>'
        else:
            _inport = '<span style="color:#bbb;">not held</span>'
        html.append(
            f'<tr style="background:{_bg};"><td><b>{_tk}</b></td>'
            f'<td>{_stage}</td><td><b>{_pos}</b></td>'
            f'<td>{_fmt_ret(_r1)}</td><td>{_fmt_ret(_r2)}</td>'
            f'<td>{_inport}</td></tr>'
        )
    html.append('</table>')
    html.append('<p style="font-size:11px; color:#888; margin-top:10px;">'
                'Returns are price-only (yfinance, 3y window). Cycle positions are '
                'qualitative reads of the shortage stage, not price targets. '
                '"In portfolio" shows the wave/sleeve a name occupies in the selected '
                'allocation; "not held" names are bottleneck candidates for reference.</p>')
    html.append('</div>')

# --- TAB 4: STOCK EXPOSURE ---
html.append('<div id="po_exposure" class="po-tabcontent">')
html.append('<div class="po-header">Stock Exposure</div>')
html.append('<table class="po-table">')
html.append('<tr><th>Underlying Stock</th><th>Net Portfolio Weight</th><th>Primary Origin</th></tr>')
for stock, weight in sorted_exposure:
    sources = list(set(exposure_sources.get(stock, [])))
    weight_str = f'{weight * 100:.2f}%'
    if len(sources) > 1:
        src_lbl = f'Cross-source Overlap ({", ".join(sorted(sources))})'
        bg = '#F3E6FA'
    else:
        src = sources[0] if sources else ''
        src_lbl = source_label(src)
        bg = SOURCE_COLORS.get(src, '#FFFFFF')
    yf_link = f'<a href="https://finance.yahoo.com/quote/{stock}/" target="_blank" style="color:#1565C0;">{stock}</a>'
    html.append(f'<tr style="background:{bg};"><td><b>{yf_link}</b></td><td>{weight_str}</td><td>{src_lbl}</td></tr>')
html.append('</table></div>')

# --- TAB 4: CRYPTO ---
html.append('<div id="po_crypto" class="po-tabcontent">')
html.append('<div class="po-header">Crypto Allocation</div>')
html.append('<table class="po-table">')
html.append('<tr><th>Asset</th><th>Holdings</th><th>Price (USD)</th><th>Value (USD)</th><th>Value (CHF)</th><th>Current %</th><th>Target %</th><th>Drift</th></tr>')
for ticker, target_pct in CRYPTO_TARGET_WEIGHTS.items():
    holdings = my_current_crypto.get(ticker, 0)
    price = crypto_prices_usd.get(ticker, 0)
    val_usd = crypto_values_usd.get(ticker, 0)
    val_chf = val_usd * USD_TO_CHF
    cur_pct = val_usd / total_crypto_usd * 100 if total_crypto_usd > 0 else 0
    tgt_pct = target_pct * 100
    label = CRYPTO_LABELS.get(ticker, ticker)
    bg = CRYPTO_COLORS.get(ticker, '#FFFFFF')
    yf_link = f'<a href="https://finance.yahoo.com/quote/{ticker}/" target="_blank" style="color:#1565C0;">{ticker}</a>'
    html.append(f'<tr style="background:{bg};"><td><b>{yf_link}</b> ({label})</td><td>{holdings:,.4f}</td><td>${price:,.2f}</td><td>{fmt_money(val_usd)}</td><td>{val_chf:,.0f} CHF</td><td>{cur_pct:.1f}%</td><td>{tgt_pct:.0f}%</td>{drift_cell(cur_pct, tgt_pct)}</tr>')
html.append(f'<tr class="po-total"><td>TOTAL</td><td></td><td></td><td>{fmt_money(total_crypto_usd)}</td><td>{total_crypto_chf:,.0f} CHF</td><td>100%</td><td>100%</td><td>—</td></tr>')
html.append('</table>')
if total_crypto_usd > 0:
    chosen = max(CRYPTO_TARGET_WEIGHTS, key=lambda c: CRYPTO_TARGET_WEIGHTS[c] - (crypto_values_usd.get(c, 0) / total_crypto_usd))
    price_chf = crypto_prices_usd[chosen] * USD_TO_CHF
    units = MONTHLY_DEPOSIT_CHF / price_chf if price_chf > 0 else 0
    html.append(f'<p style="margin-top:12px; font-size:14px; color:#1a1a1a;"><b>Crypto Buy:</b> {chosen} ({CRYPTO_LABELS.get(chosen, "")}) — {units:.4f} tokens at {price_chf:,.2f} CHF = <b>{MONTHLY_DEPOSIT_CHF:,.0f} CHF</b></p>')
html.append('</div>')

# --- TAB 5: RATIONALE ---
html.append('<div id="po_rationale" class="po-tabcontent">')
html.append('<div class="po-header">Investment Rationale</div>')

_ration_css = (
    'font-size:13px; color:#1a1a1a; line-height:1.7; max-width:900px;'
)
html.append(f'<div style="{_ration_css}">')

# Shared philosophy (applies to both modes)
html.append("""
<h3 style="color:#2C3E50;">Why a high-growth, thematic portfolio?</h3>
<p>This portfolio is a deliberate, concentrated bet on the build-out of
<b>artificial intelligence</b> — arguably the largest capital-spending cycle
in technology since the internet. Rather than owning the whole market, it
targets the specific companies that supply the "picks and shovels" of that
build-out: the chips, power, data-center hardware, cloud platforms, and
software that AI runs on.</p>
<p>The trade-off is explicit: <b>higher expected return in exchange for higher
volatility</b>. These names can fall 40-60% in a correction. That is
acceptable here because (a) the time horizon is long, (b) monthly contributions
(DCA) turn volatility into an advantage by buying more units when prices fall,
and (c) position-level discipline (see the strategy tags below) decides which
dips to buy and which to leave alone.</p>

<h3 style="color:#2C3E50;">Three operating modes per holding</h3>
<ul>
<li><b>DCA</b> — profitable, durable businesses. Buy every month regardless of
price; a drop is a discount. Never sell. (e.g. cloud + mega-cap compute.)</li>
<li><b>Cycle</b> — real businesses riding a multi-year capex wave that will
eventually crest. Buy dips, but <b>take profits near the peak</b> (~2027-2029)
rather than holding forever.</li>
<li><b>Catalyst</b> — pre-revenue / binary bets. Buy <b>once</b>, small; never
average down; sell on the specific event (approval, contract, milestone).</li>
</ul>
""")

if ALLOCATION_MODE == 'ai':
    html.append("""
    <h3 style="color:#2C3E50;">Why the W1-W6 "wave" structure?</h3>
    <p>The waves follow the AI value chain from the physical layer up to the
    application layer. Each wave answers a different question, and capital
    flows through them roughly in sequence:</p>
    <table class="po-table" style="margin:10px 0;">
      <tr><th>Wave</th><th>Layer</th><th>Why it benefits from AI</th></tr>
      <tr style="background:#E3F2FD;"><td><b>W1 Silicon</b></td>
        <td>Compute</td>
        <td>The chips AI is trained and run on. The most direct beneficiary —
        but cyclical, so a diversified semi ETF anchors it.</td></tr>
      <tr style="background:#FFF8DC;"><td><b>W2 Power</b></td>
        <td>Energy</td>
        <td>AI data centers consume enormous electricity. Nuclear, grid, and
        uranium are structurally short supply — a multi-decade tailwind.</td></tr>
      <tr style="background:#E8EAF6;"><td><b>W3 DC-Infra</b></td>
        <td>Hardware</td>
        <td>Cooling, networking, interconnect, construction — the physical
        data center. Rides the capex wave directly (mostly "cycle").</td></tr>
      <tr style="background:#E6F4EA;"><td><b>W4 Cloud</b></td>
        <td>Platform</td>
        <td>The hyperscalers that rent AI compute. The most durable, highest-
        quality cohort — the DCA core of the book.</td></tr>
      <tr style="background:#FFEBEE;"><td><b>W5 Software</b></td>
        <td>Application</td>
        <td>Profitable software embedding AI (security, data, analytics).
        Captures AI demand at the top of the stack.</td></tr>
      <tr style="background:#F3E6F5;"><td><b>W6 Speculative</b></td>
        <td>Frontier</td>
        <td>Small, asymmetric bets (quantum, space, novel hardware). Capped
        at ~5% — lottery tickets, sized once.</td></tr>
    </table>
    <p><b>Why this ordering matters:</b> owning the whole chain means that
    wherever AI value accrues — whether to chipmakers, power providers, or
    software — the portfolio captures it. It also diversifies the single-point
    risk of betting on one layer (e.g. just GPUs).</p>
    <p><b>Risk shape:</b> ~58% of the book is DCA (set-and-forget core),
    ~37% is cycle (needs an exit plan around the peak), and only ~5% is
    binary speculation. The high growth comes from the cycle + speculative
    sleeves; the stability comes from the DCA core.</p>
    """)
else:
    html.append("""
    <h3 style="color:#2C3E50;">Why "Core ETF + Satellite" structure?</h3>
    <p>This is a <b>core-satellite</b> strategy. A diversified base of thematic
    ETFs provides broad, lower-variance exposure to the AI/tech theme, while
    concentrated single-stock "satellites" add targeted high-conviction bets
    that the ETFs under-weight.</p>
    <table class="po-table" style="margin:10px 0;">
      <tr><th>Sleeve</th><th>Role</th><th>Rationale</th></tr>
      <tr style="background:#E3F2FD;"><td><b>Core ETFs</b><br>(XAIX / SMHV / QDVE)</td>
        <td>Diversified base</td>
        <td>One-line exposure to AI, semiconductors, and big-tech. Reduces
        single-stock risk and provides the portfolio's stable backbone.</td></tr>
      <tr style="background:#FFF8DC;"><td><b>Nuclear</b></td>
        <td>Power satellite</td>
        <td>AI's electricity demand makes nuclear/uranium a structural
        supply-deficit play — not captured by tech ETFs.</td></tr>
      <tr style="background:#F3E6F5;"><td><b>Quantum</b></td>
        <td>Frontier satellite</td>
        <td>Asymmetric, binary bets on next-gen compute. Tiny sizing.</td></tr>
      <tr style="background:#FFEBEE;"><td><b>Cyber</b></td>
        <td>Software satellite</td>
        <td>More AI + cloud = more attack surface. Profitable platforms.</td></tr>
      <tr style="background:#E8EAF6;"><td><b>Industrial</b></td>
        <td>Infra satellite</td>
        <td>Data-center power/cooling/construction — the physical build-out.</td></tr>
      <tr style="background:#E0F7FA;"><td><b>SpecGrowth / Other</b></td>
        <td>High-beta satellites</td>
        <td>Hyper-growth names (space, interconnect, medtech) for extra upside,
        sized small.</td></tr>
    </table>
    <p><b>Why core-satellite:</b> the ETF core captures the theme broadly and
    cheaply, so a few wrong satellite picks don't sink the portfolio — while
    the satellites still allow concentrated upside where conviction is high.</p>
    """)

html.append("""
<h3 style="color:#2C3E50;">Why monthly DCA?</h3>
<p>Contributing a fixed amount each month removes the need to time the market.
It mechanically buys more when prices are low and less when high, smooths entry
valuation, and converts the portfolio's volatility from a risk into a feature.
The drift column on each tab shows where this month's contribution should go —
toward whichever slice is most under its target.</p>
""")
html.append('</div></div>')

# =========================================================================
# 6. RENDER
# =========================================================================

display(HTML('\n'.join(html)))
